In [1]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/

In [2]:
IMG_HEIGHT = 100
IMG_WIDTH = 100
BATCH_SIZE = 16
NUM_CLASSES = 4

In [3]:
DATASET_DIR = "../Final"

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2
)

In [5]:
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

Found 1451 images belonging to 4 classes.


In [6]:
val_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 361 images belonging to 4 classes.


In [7]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)

In [8]:
for layer in base_model.layers:
    layer.trainable = False

In [9]:
model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

In [10]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,782,660 (105.98 MB)

 Trainable params: 4,194,948 (16.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [12]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_labels = list(train_generator.class_indices.keys())
print("Classes:", class_labels)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weights_dict = dict(enumerate(class_weights))
print("Class weights:", class_weights_dict)

Classes: ['Good', 'Moderate', 'Severe', 'Unhealthy']
Class weights: {0: np.float64(25.910714285714285), 1: np.float64(0.9253826530612245), 2: np.float64(1.8413705583756346), 3: np.float64(0.42777122641509435)}


In [13]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=3,
    class_weight=class_weights_dict
)

Epoch 1/3
91/91 ━━━━━━━━━━━━━━━━━━━━ 44s 393ms/step - accuracy: 0.3019 - loss: 7.8550 - val_accuracy: 0.1357 - val_loss: 1.3839
Epoch 2/3
91/91 ━━━━━━━━━━━━━━━━━━━━ 41s 449ms/step - accuracy: 0.1358 - loss: 1.3866 - val_accuracy: 0.1357 - val_loss: 1.3858
Epoch 3/3
91/91 ━━━━━━━━━━━━━━━━━━━━ 41s 453ms/step - accuracy: 0.1661 - loss: 1.3868 - val_accuracy: 0.1357 - val_loss: 1.3837


In [14]:
model.save("aqi_model_class_weight_only.h5")
print("Model saved successfully")

Model saved successfully
